# Model 7: BERT-style Prefix LM trained with DiLoCo (3 domain replicas + periodic merging)

**From the report (Sec. 4.1, App. B):**
- 12 layers, hidden 768, 12 heads, FFN 3072, **post-LayerNorm**, **GELU**, 512 learned positions, 30,522 uncased WordPiece vocab (about 110.10M parameters, tied embeddings counted once).
- Prefix mask in the spirit of UniLM: bidirectional attention over the prefix, causal over the suffix.
- Three replicas start from the **same random weights**: Supreme Court, High Courts, Acts + Tribunals.
- Each replica runs **500 local AdamW steps**, then a **Nesterov outer update** merges the parameter deltas:
  `delta_k = theta_g - theta_k`, `delta_bar = mean_k(delta_k)`, `v = mu*v + delta_bar`, `theta_g = theta_g - lr_out*(delta_bar + mu*v)`.
- The merged weights start the next round; each replica **keeps its own optimizer state**.
- About 4B aggregate training tokens, AdamW, fp16 mixed precision, clip 1.0, 1000 warmup steps, cosine decay in the final 10%.

**Not stated in the report (my choices, all in the config cell):** inner LR / betas / weight decay, outer LR and momentum (DiLoCo defaults 0.7 / 0.9), batch size, prefix-length range, block layout, the exact pretraining loss (here: next-token loss on the suffix, as in your M3 notebook), the domain token caps.

**Workflow:** (1) run sections 1-6 once to build the three domain token bins, optionally upload them as a Kaggle dataset; (2) run section 9 to train; it is resumable, so re-run it in later sessions; (3) run sections 10-11 to inspect and export.

Set `DEBUG = True` first for a 2-round synthetic-data smoke test of the whole pipeline (no downloads).

## 1. Install & login

In [ ]:
!pip install -q ijson
import os
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in.")
else:
    print("No HF token found; only public downloads will work.")

## 2. Check GPU

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = device.type == "cuda"
print("Device:", device)

## 3. Configuration

In [ ]:
import os, json, math, time, random, shutil, glob, re
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

DEBUG = False   # True = tiny synthetic-data smoke test (2 rounds x 10 local steps)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if USE_CUDA: torch.cuda.manual_seed_all(SEED)
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# ---------- Model 7 architecture (report Sec. 4.1, App. B.1) ----------
TOKENIZER_NAME = "bert-base-uncased"      # 30,522 uncased WordPieces (tokenizer only; no pretrained weights are used)
VOCAB_SIZE = 30522; D_MODEL = 768; N_LAYERS = 12; N_HEADS = 12; D_FF = 3072
HEAD_DIM = D_MODEL // N_HEADS
MAX_POS = 512; DROPOUT = 0.1; LN_EPS = 1e-12
BLOCK_SIZE = 512                          # [CLS] + 511 text tokens
PREFIX_FRAC_RANGE = (0.50, 0.75)          # per-sequence prefix length ~ U(0.50, 0.75) * BLOCK_SIZE; loss on the suffix only

# ---------- DiLoCo (report Sec. 4.1, App. B.2) ----------
REPLICAS = ["supreme_court", "high_courts", "acts_tribunals"]
N_REPLICAS = len(REPLICAS)
LOCAL_STEPS = 500                         # local AdamW steps per replica per round
OUTER_LR = 0.7; OUTER_MOMENTUM = 0.9      # DiLoCo defaults (report does not give values)
TOTAL_TOKENS = 4_000_000_000              # aggregate over all replicas
MICRO_BATCH = 16; GRAD_ACCUM = 4          # 16*4*512 = 32,768 tokens per local step

# ---------- inner optimiser ----------
PEAK_LR = 1.5e-4; MIN_LR = 1.5e-5
WARMUP_STEPS = 1000; DECAY_FRAC = 0.10    # linear warmup, constant, cosine decay in the last 10% of local steps
WEIGHT_DECAY = 0.01; BETAS = (0.9, 0.95); GRAD_CLIP = 1.0

# ---------- eval / session control ----------
VAL_BATCHES = 20; REPLICA_VAL_BATCHES = 10
SESSION_BUDGET_HOURS = 11.0               # stop cleanly (after a finished replica unit) before Kaggle's 12h limit
MAX_ROUNDS_THIS_RUN = None                # e.g. 5 to do at most 5 merges per session

TOKENS_PER_STEP = MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE
N_ROUNDS = (TOTAL_TOKENS // (N_REPLICAS * TOKENS_PER_STEP)) // LOCAL_STEPS
if DEBUG:
    LOCAL_STEPS = 10; N_ROUNDS = 2; WARMUP_STEPS = 5
    MICRO_BATCH = 4; GRAD_ACCUM = 1; VAL_BATCHES = 3; REPLICA_VAL_BATCHES = 2
    TOKENS_PER_STEP = MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE
TOTAL_INNER_STEPS = N_ROUNDS * LOCAL_STEPS   # per replica

# ---------- paths ----------
KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else os.getcwd()
SUFFIX = "_debug" if DEBUG else ""
OUT_DIR = os.path.join(WORK, "m7_diloco" + SUFFIX)
DATA_DIR = os.path.join(WORK, "m7_data" + SUFFIX)   # after uploading the bins as a Kaggle dataset, point this at /kaggle/input/.../
RAW_DIR = os.path.join(WORK, "raw_json")
RESUME_DIR = None   # previous session's OUT_DIR uploaded as a Kaggle dataset, e.g. "/kaggle/input/m7-diloco-state/m7_diloco"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"tokens/local step = {TOKENS_PER_STEP:,} | rounds = {N_ROUNDS} | local steps/replica = {TOTAL_INNER_STEPS:,}")
print(f"aggregate tokens = {N_ROUNDS * LOCAL_STEPS * N_REPLICAS * TOKENS_PER_STEP / 1e9:.2f}B")

## 4. Tokenizer
Report: Model 7 uses 30,522 uncased WordPieces (exactly the `bert-base-uncased` vocabulary). Only the tokenizer is used, the weights are random.

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
tokenizer.model_max_length = int(1e9)   # we chunk ourselves; silence the length warning
assert len(tokenizer) == VOCAB_SIZE, len(tokenizer)
CLS_ID, SEP_ID, PAD_ID, MASK_ID = tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id, tokenizer.mask_token_id
print("vocab", len(tokenizer), "| CLS", CLS_ID, "SEP", SEP_ID, "PAD", PAD_ID)

## 5. Build per-domain token bins (run once, then upload as a Kaggle dataset)
One flat `uint16` token stream per domain (documents separated by `[SEP]`):
`supreme_court` = Supreme Court judgments, `high_courts` = High Court judgments, `acts_tribunals` = Acts + Tribunals.
`FILE_SAMPLING[file] = (keep_probability, token_cap)`. The defaults roughly give each replica about 1.3-1.4B tokens (Fig. 1 of the report estimates ~0.3B SC, ~2.6B HC, ~0.17B Acts, ~2.0B Tribunals); lower them if tokenizing or disk is a problem.
Replicas with small bins (Supreme Court) simply loop over their data several times.

In [ ]:
import ijson
from huggingface_hub import hf_hub_download

DOMAIN_FILES = {
    "supreme_court":  ["Phase_1_Pretraining/Train/train_supreme_court.json"],
    "high_courts":    ["Phase_1_Pretraining/Train/train_high_courts.json"],
    "acts_tribunals": ["Phase_1_Pretraining/Train/train_acts.json",
                       "Phase_1_Pretraining/Train/train_tribunals.json"],
}
FILE_SAMPLING = {   # (keep_prob, token_cap)
    "train_supreme_court.json": (1.0, None),
    "train_high_courts.json":   (0.5, 1_400_000_000),
    "train_acts.json":          (1.0, None),
    "train_tribunals.json":     (0.6, 1_200_000_000),
}
VAL_FRAC = 0.01; VAL_MAX_TOKENS = 3_000_000; ENC_BATCH = 256; DELETE_RAW = True

def bin_paths(domain):
    return os.path.join(DATA_DIR, f"{domain}_train.bin"), os.path.join(DATA_DIR, f"{domain}_val.bin")

def encode_docs(texts):
    enc = tokenizer(texts, add_special_tokens=False, return_attention_mask=False,
                    return_token_type_ids=False, truncation=False)["input_ids"]
    return np.concatenate([np.asarray(ids + [SEP_ID], dtype=np.uint16) for ids in enc])

def flush(texts, fh):
    if not texts: return 0
    arr = encode_docs(texts); arr.tofile(fh); return len(arr)

def build_domain(domain):
    tr_path, va_path = bin_paths(domain)
    rng = random.Random(SEED)
    n_tr = n_va = 0
    with open(tr_path, "wb") as ftr, open(va_path, "wb") as fva:
        for rel in DOMAIN_FILES[domain]:
            name = os.path.basename(rel)
            keep_prob, cap = FILE_SAMPLING[name]
            path = hf_hub_download(HF_DATASET_REPO, rel, repo_type="dataset", local_dir=RAW_DIR)
            tr_texts, va_texts, file_tr = [], [], 0
            with open(path, "rb") as f:
                for rec in tqdm(ijson.items(f, "item"), desc=f"{domain}/{name}", unit=" docs"):
                    text = rec.get("text", "") if isinstance(rec, dict) else ""
                    if not text or not text.strip(): continue
                    if rng.random() < VAL_FRAC:
                        if n_va < VAL_MAX_TOKENS: va_texts.append(text)
                    elif rng.random() < keep_prob:
                        tr_texts.append(text)
                    if len(tr_texts) >= ENC_BATCH:
                        n = flush(tr_texts, ftr); n_tr += n; file_tr += n; tr_texts = []
                    if len(va_texts) >= 64:
                        n_va += flush(va_texts, fva); va_texts = []
                    if cap and file_tr >= cap: break
            n = flush(tr_texts, ftr); n_tr += n; file_tr += n
            n_va += flush(va_texts, fva)
            print(f"  {name}: {file_tr/1e6:.0f}M train tokens kept")
            if DELETE_RAW and os.path.exists(path): os.remove(path)
    return n_tr, n_va

def build_synthetic():
    rng = np.random.default_rng(0)
    for d in REPLICAS:
        tr, va = bin_paths(d)
        rng.integers(1000, VOCAB_SIZE, 400_000).astype(np.uint16).tofile(tr)
        rng.integers(1000, VOCAB_SIZE, 60_000).astype(np.uint16).tofile(va)

os.makedirs(DATA_DIR, exist_ok=True)
if all(os.path.exists(p) for d in REPLICAS for p in bin_paths(d)):
    print("Token bins found in", DATA_DIR)
elif DEBUG:
    build_synthetic(); print("Synthetic bins written (DEBUG).")
else:
    meta = {}
    for d in REPLICAS:
        n_tr, n_va = build_domain(d)
        meta[d] = {"train_tokens": n_tr, "val_tokens": n_va}
    json.dump(meta, open(os.path.join(DATA_DIR, "meta.json"), "w"), indent=1)

need_per_replica = TOTAL_INNER_STEPS * TOKENS_PER_STEP
for d in REPLICAS:
    n = os.path.getsize(bin_paths(d)[0]) // 2
    print(f"{d:15s} {n/1e6:8.0f}M train tokens -> ~{need_per_replica/max(n,1):.2f} passes over the data")

## 6. Data streams
Each replica owns a deterministic, resumable shuffled stream over its own domain bin. Every sequence is `[CLS]` + 511 tokens.

In [ ]:
class DomainStream:
    def __init__(self, path, seed, block_len=BLOCK_SIZE - 1):
        self.arr = np.memmap(path, dtype=np.uint16, mode="r")
        self.block_len = block_len; self.seed = seed
        self.n_blocks = len(self.arr) // block_len
        assert self.n_blocks > 0, f"{path} is too small"
        self.epoch = 0; self.pos = 0; self._perm = None
    def _make_perm(self):
        self._perm = np.random.default_rng(self.seed + 1_000_003 * self.epoch).permutation(self.n_blocks)
    def next_batch(self, bs):
        if self._perm is None: self._make_perm()
        rows = []
        for _ in range(bs):
            if self.pos >= self.n_blocks:
                self.epoch += 1; self.pos = 0; self._make_perm()
            s = int(self._perm[self.pos]) * self.block_len; self.pos += 1
            rows.append(self.arr[s:s + self.block_len])
        x = np.empty((bs, self.block_len + 1), dtype=np.int64)
        x[:, 0] = CLS_ID; x[:, 1:] = np.stack(rows)
        return torch.from_numpy(x)
    def state_dict(self): return {"epoch": self.epoch, "pos": self.pos}
    def load_state_dict(self, s): self.epoch, self.pos, self._perm = s["epoch"], s["pos"], None

def load_val_batches(domain, n_batches, bs):
    arr = np.memmap(bin_paths(domain)[1], dtype=np.uint16, mode="r")
    L = BLOCK_SIZE - 1; nb = len(arr) // L
    idx = np.unique(np.linspace(0, nb - 1, num=min(n_batches * bs, nb), dtype=np.int64))
    idx = idx[: (len(idx) // bs) * bs]
    out = []
    for i in range(0, len(idx), bs):
        x = np.empty((bs, L + 1), dtype=np.int64); x[:, 0] = CLS_ID
        for j, b in enumerate(idx[i:i + bs]): x[j, 1:] = arr[b * L:(b + 1) * L]
        out.append(torch.from_numpy(x))
    return out

VAL = {d: load_val_batches(d, VAL_BATCHES, MICRO_BATCH) for d in REPLICAS}
print({d: len(v) for d, v in VAL.items()}, "val batches per domain")

## 7. Model: BERT-style prefix LM
Post-LN encoder with GELU and learned absolute positions. Module names mirror `transformers.BertModel` / `BertForMaskedLM`, so `model.bert.*` weights load straight into a Hugging Face `BertModel` (section 11).
- Attention mask: query `i` may see key `j` if `j < prefix_len` (bidirectional prefix) or `j <= i` (causal suffix).
- Loss: next-token prediction on suffix positions only (position `j` predicts token `j+1`, supervised when `j >= prefix_len - 1`).
- `prefix_lens=None` means fully bidirectional, which is how the model is used as an encoder for downstream tasks (P3-P7).
- The pooler is unused in pretraining; it is kept because it is part of the BERT layout and brings the count to the report's 110.10M.

In [ ]:
def prefix_mask(prefix_lens, T):
    idx = torch.arange(T, device=prefix_lens.device)
    causal = idx[None, :] <= idx[:, None]                        # (T,T)
    in_prefix = idx[None, None, :] < prefix_lens[:, None, None]  # (B,1,T) keys inside the prefix
    return (causal[None] | in_prefix)[:, None]                   # (B,1,T,T) bool, True = may attend

class BertEmbeddings(nn.Module):
    def __init__(self):
        super().__init__()
        self.word_embeddings = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.position_embeddings = nn.Embedding(MAX_POS, D_MODEL)
        self.token_type_embeddings = nn.Embedding(2, D_MODEL)
        self.LayerNorm = nn.LayerNorm(D_MODEL, eps=LN_EPS)
        self.dropout = nn.Dropout(DROPOUT)
    def forward(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device)
        x = self.word_embeddings(ids) + self.position_embeddings(pos)[None] + self.token_type_embeddings.weight[0]
        return self.dropout(self.LayerNorm(x))

class BertSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.query = nn.Linear(D_MODEL, D_MODEL); self.key = nn.Linear(D_MODEL, D_MODEL); self.value = nn.Linear(D_MODEL, D_MODEL)
    def forward(self, x, mask):
        B, T, _ = x.shape
        sh = lambda t: t.view(B, T, N_HEADS, HEAD_DIM).transpose(1, 2)
        q, k, v = sh(self.query(x)), sh(self.key(x)), sh(self.value(x))
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=DROPOUT if self.training else 0.0)
        return o.transpose(1, 2).reshape(B, T, D_MODEL)

class BertSelfOutput(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.Linear(D_MODEL, D_MODEL); self.LayerNorm = nn.LayerNorm(D_MODEL, eps=LN_EPS); self.dropout = nn.Dropout(DROPOUT)
    def forward(self, h, residual): return self.LayerNorm(self.dropout(self.dense(h)) + residual)

class BertAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.self = BertSelfAttention(); self.output = BertSelfOutput()
    def forward(self, x, mask): return self.output(self.self(x, mask), x)

class BertIntermediate(nn.Module):
    def __init__(self):
        super().__init__(); self.dense = nn.Linear(D_MODEL, D_FF)
    def forward(self, x): return F.gelu(self.dense(x))

class BertOutput(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense = nn.Linear(D_FF, D_MODEL); self.LayerNorm = nn.LayerNorm(D_MODEL, eps=LN_EPS); self.dropout = nn.Dropout(DROPOUT)
    def forward(self, h, residual): return self.LayerNorm(self.dropout(self.dense(h)) + residual)

class BertLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = BertAttention(); self.intermediate = BertIntermediate(); self.output = BertOutput()
    def forward(self, x, mask):
        a = self.attention(x, mask)
        return self.output(self.intermediate(a), a)

class BertEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.layer = nn.ModuleList([BertLayer() for _ in range(N_LAYERS)])

class BertPooler(nn.Module):
    def __init__(self):
        super().__init__(); self.dense = nn.Linear(D_MODEL, D_MODEL)
    def forward(self, h): return torch.tanh(self.dense(h[:, 0]))

class PrefixBertModel(nn.Module):
    """Same layout as transformers.BertModel; attention mask is a prefix mask."""
    def __init__(self):
        super().__init__()
        self.embeddings = BertEmbeddings(); self.encoder = BertEncoder(); self.pooler = BertPooler()
    def forward(self, ids, prefix_lens=None):
        x = self.embeddings(ids)
        mask = None if prefix_lens is None else prefix_mask(prefix_lens, ids.size(1))
        for layer in self.encoder.layer: x = layer(x, mask)
        return x

class PredictionTransform(nn.Module):
    def __init__(self):
        super().__init__(); self.dense = nn.Linear(D_MODEL, D_MODEL); self.LayerNorm = nn.LayerNorm(D_MODEL, eps=LN_EPS)
    def forward(self, h): return self.LayerNorm(F.gelu(self.dense(h)))

class PredictionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.transform = PredictionTransform()
        self.decoder = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)   # weight tied to word embeddings
        self.bias = nn.Parameter(torch.zeros(VOCAB_SIZE))
    def forward(self, h): return self.decoder(self.transform(h)) + self.bias

class LMHead(nn.Module):
    def __init__(self):
        super().__init__(); self.predictions = PredictionHead()

class PrefixBertForLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = PrefixBertModel(); self.cls = LMHead()
        self.apply(self._init)
        self.cls.predictions.decoder.weight = self.bert.embeddings.word_embeddings.weight   # tie
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)
    def forward(self, ids, prefix_lens):
        h = self.bert(ids, prefix_lens)                              # (B,T,D)
        T = ids.size(1)
        pos = torch.arange(T - 1, device=ids.device)
        sup = pos[None, :] >= (prefix_lens[:, None] - 1)             # supervised positions (B,T-1)
        logits = self.cls.predictions(h[:, :-1][sup])                # only suffix positions hit the vocab matmul
        tgt = ids[:, 1:][sup]
        return F.cross_entropy(logits.float(), tgt), tgt.numel()

model = PrefixBertForLM().to(device)
params = list(model.parameters())                                    # de-duplicated, stable order (tied weight once)
param_names = [n for n, _ in model.named_parameters()]
n_params = sum(p.numel() for p in params)
print(f"Model 7 parameters (tied weights counted once): {n_params:,} ({n_params/1e6:.2f}M)")
assert 110_000_000 <= n_params <= 110_200_000

## 8. Optimizer, LR schedule, DiLoCo state and helpers
- One model object is reused for all replicas. Each replica has its **own AdamW (and GradScaler) state** and data stream; AdamW states stay on the GPU (about 0.9 GB each).
- Global state = `theta_g` (merged weights) + `outer_mom` (outer Nesterov momentum), both fp32.
- Every replica unit begins by copying `theta_g` into the model and ends with `delta = theta_g - theta_k`.
- After each finished replica unit its optimizer/stream state **and delta** are written to `replica{k}_r{rounds_done}.pt`; after each merge `global.pt` is written. This makes the run resumable at replica granularity.

In [ ]:
class WSDScheduler:
    def __init__(self, peak_lr, min_lr, warmup_steps, total_steps, decay_frac=0.10):
        self.peak_lr, self.min_lr, self.warmup_steps, self.total_steps = peak_lr, min_lr, warmup_steps, total_steps
        self.decay_start = int(total_steps * (1 - decay_frac)); self.decay_steps = total_steps - self.decay_start
    def get_lr(self, step):
        if step < self.warmup_steps: return self.peak_lr * (step + 1) / max(self.warmup_steps, 1)
        if step < self.decay_start: return self.peak_lr
        p = min((step - self.decay_start) / max(self.decay_steps, 1), 1.0)
        return self.min_lr + (self.peak_lr - self.min_lr) * 0.5 * (1 + math.cos(math.pi * p))
sched = WSDScheduler(PEAK_LR, MIN_LR, WARMUP_STEPS, TOTAL_INNER_STEPS, DECAY_FRAC)

decay_p, no_decay_p = [], []
for n, p in model.named_parameters():
    (no_decay_p if ("bias" in n or "LayerNorm" in n) else decay_p).append(p)

class Replica:
    def __init__(self, k, name):
        self.k, self.name = k, name
        self.stream = DomainStream(bin_paths(name)[0], seed=SEED + 17 * k)
        self.opt = torch.optim.AdamW([{"params": decay_p, "weight_decay": WEIGHT_DECAY},
                                      {"params": no_decay_p, "weight_decay": 0.0}],
                                     lr=PEAK_LR, betas=BETAS, fused=USE_CUDA)
        self.scaler = torch.amp.GradScaler("cuda", enabled=USE_CUDA)
        self.inner_step = 0; self.rounds_done = 0; self.tokens = 0
        self.delta = None   # only filled when loaded from disk for an already-finished unit

replicas = [Replica(k, n) for k, n in enumerate(REPLICAS)]
theta_g = [p.detach().clone() for p in params]
outer_mom = [torch.zeros_like(p) for p in params]
global_round = 0

def autocast(): return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_CUDA)
def load_theta_into_model():
    with torch.no_grad():
        for p, g in zip(params, theta_g): p.copy_(g)

@torch.no_grad()
def eval_domain(domain, n_batches):
    model.eval(); tot, cnt = 0.0, 0
    for b in VAL[domain][:n_batches]:
        b = b.to(device)
        plen = torch.full((b.size(0),), int(0.75 * b.size(1)), device=device, dtype=torch.long)
        with autocast(): loss, n = model(b, plen)
        tot += loss.item() * n; cnt += n
    model.train()
    return tot / max(cnt, 1)

@torch.no_grad()
def outer_step(deltas):
    """DiLoCo outer update (Nesterov SGD on the averaged delta): v = mu*v + dbar ; theta = theta - lr*(dbar + mu*v)."""
    for i, (g, v) in enumerate(zip(theta_g, outer_mom)):
        dbar = deltas[0][i].clone()
        for d in deltas[1:]: dbar.add_(d[i])
        dbar.div_(len(deltas))
        v.mul_(OUTER_MOMENTUM).add_(dbar)
        g.sub_(OUTER_LR * (dbar + OUTER_MOMENTUM * v))

@torch.no_grad()
def delta_stats(deltas):
    K = len(deltas); dots = torch.zeros(K, K, dtype=torch.float64, device=device)
    for i in range(len(params)):
        flat = [d[i].flatten().double() for d in deltas]
        for a in range(K):
            for b in range(a, K):
                dots[a, b] += torch.dot(flat[a], flat[b])
    dots = dots + dots.triu(1).T
    norms = dots.diag().sqrt()
    cos = dots / (norms[:, None] * norms[None, :] + 1e-12)
    return norms.tolist(), cos.tolist()

# ----- persistence -----
def replica_file(k, R): return os.path.join(OUT_DIR, f"replica{k}_r{R}.pt")
def global_file(): return os.path.join(OUT_DIR, "global.pt")
LOG_PATH = os.path.join(OUT_DIR, "train_log.json")

def atomic_save(obj, path):
    tmp = path + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def save_replica(rep, delta):
    atomic_save({"opt": rep.opt.state_dict(), "scaler": rep.scaler.state_dict(), "stream": rep.stream.state_dict(),
                 "inner_step": rep.inner_step, "rounds_done": rep.rounds_done, "tokens": rep.tokens,
                 "delta": [d.cpu() for d in delta]}, replica_file(rep.k, rep.rounds_done))
    for f in glob.glob(os.path.join(OUT_DIR, f"replica{rep.k}_r*.pt")):
        if int(re.search(r"_r(\d+)\.pt$", f).group(1)) < rep.rounds_done: os.remove(f)

def save_global(round_done):
    atomic_save({"theta": [t.cpu() for t in theta_g], "momentum": [m.cpu() for m in outer_mom],
                 "round": round_done, "param_names": param_names}, global_file())

def restore_from_resume_dir():
    if RESUME_DIR and os.path.isdir(RESUME_DIR) and not os.path.exists(global_file()) \
       and not glob.glob(os.path.join(OUT_DIR, "replica*_r*.pt")):
        print("Copying state from", RESUME_DIR)
        for f in glob.glob(os.path.join(RESUME_DIR, "*")):
            if os.path.isfile(f) and not f.endswith(".tmp"): shutil.copy(f, OUT_DIR)

def load_state():
    global global_round
    restore_from_resume_dir()
    if os.path.exists(global_file()):
        ck = torch.load(global_file(), map_location="cpu", weights_only=False)
        assert ck["param_names"] == param_names, "parameter layout changed since the checkpoint was written"
        for g, t in zip(theta_g, ck["theta"]): g.copy_(t)
        for m, t in zip(outer_mom, ck["momentum"]): m.copy_(t)
        global_round = ck["round"]
        print(f"Loaded global state: {global_round} merges done")
    for rep in replicas:
        files = glob.glob(os.path.join(OUT_DIR, f"replica{rep.k}_r*.pt"))
        if not files: continue
        best = max(files, key=lambda f: int(re.search(r"_r(\d+)\.pt$", f).group(1)))
        ck = torch.load(best, map_location="cpu", weights_only=False)
        rep.opt.load_state_dict(ck["opt"]); rep.scaler.load_state_dict(ck["scaler"]); rep.stream.load_state_dict(ck["stream"])
        rep.inner_step, rep.rounds_done, rep.tokens = ck["inner_step"], ck["rounds_done"], ck["tokens"]
        assert rep.rounds_done <= global_round + 1, f"replica {rep.k} is ahead of the global state"
        if rep.rounds_done == global_round + 1:   # unit finished, merge still pending
            rep.delta = [d.to(device) for d in ck["delta"]]
        print(f"  replica {rep.k} ({rep.name}): {rep.rounds_done} rounds done, step {rep.inner_step}, stream epoch {rep.stream.epoch}")
        del ck

load_state()

## 9. Train: local steps on each replica, then merge, repeat
Per round: for each replica, load `theta_g`, run 500 local AdamW steps on that replica's domain, store `delta`; then apply the outer Nesterov update and evaluate the merged model on all three domain validation sets.
**Resuming:** just re-run this cell. Already finished replica units of the current round are skipped and their stored deltas reused. For a new Kaggle session, save `OUT_DIR` as a dataset, set `RESUME_DIR`, and re-run sections 1-9.

In [ ]:
lo_p, hi_p = int(PREFIX_FRAC_RANGE[0] * BLOCK_SIZE), int(PREFIX_FRAC_RANGE[1] * BLOCK_SIZE)

def run_unit(rep, r):
    load_theta_into_model()
    model.train()
    acc = torch.zeros((), device=device); t0 = time.time()
    for i in range(LOCAL_STEPS):
        lr = sched.get_lr(rep.inner_step)
        for g in rep.opt.param_groups: g["lr"] = lr
        rep.opt.zero_grad(set_to_none=True)
        step_loss = torch.zeros((), device=device)
        for _ in range(GRAD_ACCUM):
            batch = rep.stream.next_batch(MICRO_BATCH).to(device, non_blocking=True)
            plen = torch.randint(lo_p, hi_p + 1, (MICRO_BATCH,), device=device)
            with autocast(): loss, _ = model(batch, plen)
            rep.scaler.scale(loss / GRAD_ACCUM).backward()
            step_loss += loss.detach() / GRAD_ACCUM
        rep.scaler.unscale_(rep.opt)
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        rep.scaler.step(rep.opt); rep.scaler.update()
        acc += step_loss; rep.inner_step += 1; rep.tokens += TOKENS_PER_STEP
        if (i + 1) % 100 == 0 or i == LOCAL_STEPS - 1:
            n = (i % 100) + 1
            cur = acc.item() / n; acc.zero_()
            if not math.isfinite(cur): raise RuntimeError("Loss is NaN/inf: lower PEAK_LR or check the data.")
            tqdm.write(f"  r{r+1} [{rep.name}] step {i+1}/{LOCAL_STEPS} | loss {cur:.4f} | lr {lr:.2e} | {rep.tokens/1e6:.0f}M tok | {time.time()-t0:.0f}s")
            last_loss = cur
    delta = [g - p.detach() for g, p in zip(theta_g, params)]     # delta_k = theta_g - theta_k
    own_val = eval_domain(rep.name, REPLICA_VAL_BATCHES)           # replica on its own domain, before merging
    return delta, last_loss, own_val

log = json.load(open(LOG_PATH)) if os.path.exists(LOG_PATH) else []
best_val = min([e["val_avg"] for e in log], default=float("inf"))
T0 = time.time(); unit_secs = 0.0; merges_this_run = 0; stop = False

for r in range(global_round, N_ROUNDS):
    deltas = [None] * N_REPLICAS; tr_loss = {}; own_val = {}
    for rep in replicas:
        if rep.rounds_done >= r + 1:                 # finished earlier (resume): reuse stored delta
            deltas[rep.k] = rep.delta; continue
        if (time.time() - T0) + 1.15 * unit_secs > SESSION_BUDGET_HOURS * 3600:
            print("Session budget reached: stopping cleanly. Save OUT_DIR and resume in a new session."); stop = True; break
        ts = time.time()
        d, tr_loss[rep.name], own_val[rep.name] = run_unit(rep, r)
        rep.rounds_done = r + 1
        save_replica(rep, d)
        deltas[rep.k] = d
        unit_secs = time.time() - ts
        print(f"Round {r+1}/{N_ROUNDS} replica {rep.name}: train {tr_loss[rep.name]:.4f}, own-domain val {own_val[rep.name]:.4f} ({unit_secs/60:.1f} min)")
    if stop: break

    norms, cos = delta_stats(deltas)
    outer_step(deltas)
    save_global(r + 1)
    for rep in replicas: rep.delta = None
    del deltas

    load_theta_into_model()
    val = {d: eval_domain(d, VAL_BATCHES) for d in REPLICAS}
    val_avg = float(np.mean(list(val.values())))
    entry = {"round": r + 1, "inner_steps_per_replica": (r + 1) * LOCAL_STEPS,
             "tokens_total": sum(rep.tokens for rep in replicas), "val": val, "val_avg": val_avg,
             "train_loss": tr_loss, "own_val_premerge": own_val, "delta_norm": norms, "delta_cos": cos}
    log.append(entry); json.dump(log, open(LOG_PATH, "w"), indent=1)
    print(f"=== MERGED after round {r+1}: " + " | ".join(f"{d} {v:.3f}" for d, v in val.items()) + f" | avg {val_avg:.4f} | delta cos(0,1)={cos[0][1]:.2f} cos(0,2)={cos[0][2]:.2f} cos(1,2)={cos[1][2]:.2f}")
    if val_avg < best_val:
        best_val = val_avg
        torch.save({"model_state_dict": model.state_dict(), "round": r + 1, "val_avg": val_avg}, os.path.join(OUT_DIR, "best_merged.pt"))
    global_round = r + 1; merges_this_run += 1
    if MAX_ROUNDS_THIS_RUN and merges_this_run >= MAX_ROUNDS_THIS_RUN:
        print("MAX_ROUNDS_THIS_RUN reached."); break

print(f"\nStopped at {global_round}/{N_ROUNDS} merges, elapsed {(time.time()-T0)/3600:.2f}h")

## 10. Inspect: merged validation loss per domain, replica specialization
`own_val_premerge` (replica on its own domain just before merging) should be lower than the merged model's loss on that domain early on (specialization); the gap should shrink as rounds progress. Low or negative `delta_cos` between replicas means they pull in different directions.

In [ ]:
import matplotlib.pyplot as plt
log = json.load(open(LOG_PATH))
if log:
    rounds = [e["round"] for e in log]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for d in REPLICAS: ax[0].plot(rounds, [e["val"][d] for e in log], marker="o", ms=3, label=f"merged / {d}")
    ax[0].plot(rounds, [e["val_avg"] for e in log], "k--", label="avg"); ax[0].set_xlabel("round"); ax[0].set_ylabel("val loss (suffix NLL)"); ax[0].legend(); ax[0].grid(alpha=.3)
    for a, b in [(0, 1), (0, 2), (1, 2)]: ax[1].plot(rounds, [e["delta_cos"][a][b] for e in log], label=f"cos({REPLICAS[a]}, {REPLICAS[b]})")
    ax[1].set_xlabel("round"); ax[1].set_ylabel("cosine of replica deltas"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "diloco_curves.png"), dpi=150); plt.show()

## 11. Export the merged Model 7
Saves (a) a plain PyTorch checkpoint of the merged weights (`m7_merged.pt`), and (b) a Hugging Face `BertModel` folder (`m7_hf_bert/`) with tokenizer for the downstream phases. Use the final merged weights (`theta_g`) unless `best_merged.pt` has a clearly lower validation loss.

**Using Model 7 downstream (P3-P7):** feed the whole input as the prefix (fully bidirectional), i.e. `model.bert(ids, prefix_lens=None)` in this notebook's classes, or the Hugging Face `BertModel` with a normal 2D attention mask.

In [ ]:
load_theta_into_model()
cfg = dict(arch="bert_prefix_lm", vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, d_ff=D_FF,
           max_pos=MAX_POS, tokenizer=TOKENIZER_NAME, merged_rounds=global_round)
torch.save({"model_state_dict": model.state_dict(), "config": cfg}, os.path.join(OUT_DIR, "m7_merged.pt"))
print("Saved", os.path.join(OUT_DIR, "m7_merged.pt"))

try:
    from transformers import BertConfig, BertModel
    hf_cfg = BertConfig(vocab_size=VOCAB_SIZE, hidden_size=D_MODEL, num_hidden_layers=N_LAYERS, num_attention_heads=N_HEADS,
                        intermediate_size=D_FF, max_position_embeddings=MAX_POS, hidden_act="gelu", hidden_dropout_prob=DROPOUT,
                        attention_probs_dropout_prob=DROPOUT, layer_norm_eps=LN_EPS, type_vocab_size=2)
    hf = BertModel(hf_cfg)
    sd = {k[len("bert."):]: v.cpu() for k, v in model.state_dict().items() if k.startswith("bert.")}
    missing, unexpected = hf.load_state_dict(sd, strict=False)
    print("HF load -> missing:", missing, "| unexpected:", unexpected)
    # sanity: HF encoder with a 3-D prefix mask must match our encoder (fp32, eval mode)
    model.eval(); hf.to(device).eval()
    x = torch.randint(1000, VOCAB_SIZE, (2, 64), device=device); pl = torch.tensor([20, 40], device=device)
    with torch.no_grad():
        a = model.bert(x, pl)
        b = hf(input_ids=x, attention_mask=prefix_mask(pl, 64)[:, 0].long()).last_hidden_state
    print("max |ours - HF| on a prefix-masked batch:", (a - b).abs().max().item())
    hf.cpu().save_pretrained(os.path.join(OUT_DIR, "m7_hf_bert")); tokenizer.save_pretrained(os.path.join(OUT_DIR, "m7_hf_bert"))
    print("Saved", os.path.join(OUT_DIR, "m7_hf_bert"))
except Exception as e:
    print("HF export skipped:", repr(e))

## 12. Summary

In [ ]:
print("=" * 60); print("MODEL 7 / DiLoCo SUMMARY"); print("=" * 60)
print(f"Parameters: {n_params/1e6:.2f}M | {N_LAYERS}L d={D_MODEL} h={N_HEADS} ffn={D_FF} post-LN GELU, {MAX_POS} learned pos")
print(f"Replicas: {REPLICAS} | {LOCAL_STEPS} local AdamW steps/round | outer Nesterov lr={OUTER_LR} mu={OUTER_MOMENTUM}")
print(f"Merges done: {global_round}/{N_ROUNDS} | tokens seen: {sum(r.tokens for r in replicas)/1e9:.2f}B")
if log: print(f"Last merged val: " + ", ".join(f"{d} {v:.3f}" for d, v in log[-1]["val"].items()))